# 01 — Data Exploration

Initial look at the raw Telco Customer Churn dataset: shape, dtypes, missing values, duplicates, and the target variable, before any cleaning is applied.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Shape:", df.shape)


Shape: (7043, 21)


## Columns and data types

In [2]:
print(df.dtypes)

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


## First few rows

In [3]:
df.head(3)
print(df.head(3).to_string())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling     PaymentMethod  MonthlyCharges TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes  Electronic check           29.85        29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No      Mailed check           56.95       1889.5    No
2  3668-QPYBK    Male              0      No         No       2          Yes                No             DSL            Yes          Yes               N

## Missing values (standard NaN check)

In [4]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


## The hidden `TotalCharges` problem

A plain `.isnull()` check shows zero missing values, but `TotalCharges` is
stored as **text**, not a number. Coercing it to numeric reveals blank
strings that `.isnull()` cannot see.

In [5]:
blank_mask = df['TotalCharges'].astype(str).str.strip() == ''
print("Blank TotalCharges entries:", blank_mask.sum())
print(df.loc[blank_mask, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].to_string(index=False))


Blank TotalCharges entries: 11
customerID  tenure  MonthlyCharges TotalCharges Churn
4472-LVYGI       0           52.55                 No
3115-CZMZD       0           20.25                 No
5709-LVOEQ       0           80.85                 No
4367-NUYAO       0           25.75                 No
1371-DWPAZ       0           56.05                 No
7644-OMVMY       0           19.85                 No
3213-VVOLG       0           25.35                 No
2520-SGTTA       0           20.00                 No
2923-ARZLG       0           19.70                 No
4075-WKNIU       0           73.35                 No
2775-SEFEE       0           61.90                 No


Every one of these rows has `tenure == 0` — brand-new customers who have not been billed yet. This is why the cleaning step sets `TotalCharges = 0` for these rows rather than treating them as missing data.

## Duplicates

In [6]:
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())


Duplicate customerIDs: 0
Fully duplicated rows: 0


## Target variable — class distribution

In [7]:
counts = df['Churn'].value_counts()
pct = df['Churn'].value_counts(normalize=True) * 100
print(counts)
print()
print(pct.round(2))


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


## Summary

- **7,043 rows, 21 columns**, one row per customer.
- No `NaN`s reported by pandas, but `TotalCharges` hides **11 blank-string
  rows** that need explicit handling (see `src/data_preprocessing.py`).
- **0 duplicate customer IDs, 0 duplicate rows.**
- Target is imbalanced: **73.5% No-churn vs 26.5% Churn** (~2.8:1) —
  moderate imbalance, addressed with `class_weight='balanced'` rather than
  synthetic oversampling (see `notebooks/04_model_training.ipynb`).

Next: `02_eda.ipynb` — exploratory analysis of churn patterns across
customer segments.